In [13]:
import requests
import time
from config import Config

def route_query(query, chat_history=None):
    url = f"2/route"
    #url = Config.ROUTER_API
    headers = {
        "Authorization": f"Bearer {Config.API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": Config.REQUEST_ID_ROUTER,
        "Content-Type": "application/json"
    }
    data = {"query": query}
    if chat_history:
        data["chatHistory"] = chat_history

    while True:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 429:
            time.sleep(5)
            continue
        return response.json()

In [14]:
#라우터 성능 확인하기 위해 1차 확인!

query1= "고구려인 500명중 40%는 남자레, 그럼 여자는 몇명일까"
router_result1 = route_query(query1)

query2= "세종대왕의 업적은 한글창작, 인재 등용 등 매우 많다를 영어로 번역해줘"
router_result2 = route_query(query2)

# JSON 전체 보기
print(router_result1)
print(router_result2)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '역사 질문', 'called': True}, 'blockedContent': {'result': ['Mathematics'], 'called': True}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 678, 'completionTokens': 38, 'totalTokens': 716}}}
{'status': {'code': '20000', 'message': 'OK'}, 'result': {'domain': {'result': '역사 질문', 'called': True}, 'blockedContent': {'result': ['Translation'], 'called': True}, 'safety': {'result': [], 'called': False}, 'usage': {'promptTokens': 678, 'completionTokens': 38, 'totalTokens': 716}}}


In [16]:
def get_chat_response(query, chat_history=None):
    # 1. 라우터 결과 확인
    router_result = route_query(query, chat_history)

    domain = router_result.get("result", {}).get("domain", {}).get("result", "")
    blocked_content = router_result.get("result", {}).get("blockedContent", {}).get("result", [])

    # 2. 허용 도메인 목록
    ALLOW_DOMAINS = ["역사 질문"]

    # 3. 도메인 미허용 시
    if domain not in ALLOW_DOMAINS:
        return {
            "message": "역사 관련 외 질문은 답변이 불가능합니다.",
            "filtered_domain": domain or "미분류"
        }

    # 4. blockedContent 필터에 걸릴 경우
    if blocked_content:
        return {
            "message": "해당 질문은 필터로 인해 차단되었습니다.",
            "blocked_topics": blocked_content,
            "domain": domain
        }

    # 5. 통과된 경우 HCX Chat API 호출
    url = Config.CHAT_COMPLETIONS_API
    headers = {
        'Authorization': f'Bearer {Config.API_KEY}',
        'X-NCP-CLOVASTUDIO-REQUEST-ID': Config.REQUEST_ID_CHAT,
        'Content-Type': 'application/json',
    }

    system_prompt = "당신은 업무 도우미 입니다."
    messages = [{'role': 'system', 'content': system_prompt}]

    if chat_history:
        messages.extend(chat_history[-3:])
    else:
        messages.append({'role': 'user', 'content': query})

    data = {
        'messages': messages,
        "maxTokens": 512,
        "seed": 0,
        "temperature": 0.4,
        "topP": 0.4,
        "topK": 0,
        "repeatPenalty": 5.0
    }

    response = requests.post(url, headers=headers, json=data)
    return response.json()


In [ ]:
response = get_chat_response("고구려인 500명중 40%는 남자레, 그럼 여자는 몇명일까")
print(response)

{'message': '해당 질문은 필터로 인해 차단되었습니다.', 'blocked_topics': ['Mathematics'], 'domain': '역사 질문'}


In [ ]:
response = get_chat_response("세종대왕의 업적은 한글창작, 인재 등용 등 매우 많다를 영어로 번역해줘")
print(response)

{'message': '해당 질문은 필터로 인해 차단되었습니다.', 'blocked_topics': ['Translation'], 'domain': '역사 질문'}


In [ ]:
response = get_chat_response("조선시대에서도 외국어 교육이 이루어졌을까??")
print(response)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'message': {'role': 'assistant', 'content': '네, 조선시대에서도 외국어 교육이 이루어졌습니다. \n\n1.**중국어 교육**: 조선시대에는 중국과의 교류가 매우 중요했기 때문에, 중국어 교육이 가장 활발하게 이루어졌습니다. 중국어를 배우는 사람들은 주로 관료나 학자들이었으며, 이들은 중국어를 통해 중국의 문화와 역사를 이해하고, 외교 업무를 수행할 수 있었습니다.\n\n2.**일본어 교육**: 일본어 교육도 이루어졌습니다. 특히, 임진왜란 이후 일본과의 교류가 증가하면서 일본어 교육이 더욱 중요해졌습니다. 일본어를 배우는 사람들은 주로 외교관이나 상인들이었습니다.\n\n3.**몽골어 교육**: 고려시대부터 몽골과의 교류가 있었기 때문에 몽골어 교육도 이루어졌습니다. 하지만 조선시대에는 몽골과의 교류가 줄어들면서 몽골어 교육 역시 감소했습니다.\n\n4.**영어 교육**: 19세기 말부터 서양 열강들이 조선에 진출하면서 영어 교육이 시작되었습니다. 당시 정부는 영어 교육을 통해 서양 문물을 받아들이고, 국제사회에서 경쟁력을 확보하고자 했습니다.\n\n이러한 외국어 교육은 주로 궁중이나 관청에서 이루어졌으며, 개인적으로 외국어를 배우는 사람들도 있었습니다. 그러나 현대적인 의미의 외국어 교육과는 다소 차이가 있으며, 주로 일상생활에서 사용되는 회화보다는 문서 작성이나 외교 업무 등에 필요한 언어 능력을 중심으로 교육이 이루어졌습니다.'}, 'inputLength': 19, 'outputLength': 289, 'stopReason': 'stop_before', 'seed': 39335005}}


In [ ]:
response = get_chat_response("한국 남북전쟁이 휴전되고 몇년만에 4대 대통령이 당선되었지?")
print(response)

{'status': {'code': '20000', 'message': 'OK'}, 'result': {'message': {'role': 'assistant', 'content': '한국의 4대 대통령은 윤보선 전 대통령으로, 1960년 8월 13일에 당선되었습니다. 한국 전쟁이 1953년 7월 27일에 휴전 협정이 체결 되었으므로, 약 7년 만에 4대 대통령이 당선 되었습니다.\n\n더 궁금하신 점이나 다른 질문이 있으시면 언제든지 말씀해 주세요!'}, 'inputLength': 26, 'outputLength': 71, 'stopReason': 'stop_before', 'seed': 3409532723}}
